# Paso 2: del grafo único al par de grafos

Acá doy el segundo paso hacia el algoritmo para la conjetura de Stanley. La idea es
simple: **reuso el mismo algoritmo de Wagner del notebook 01 y solo cambio la
interpretación**.

En vez de que la palabra binaria represente un grafo, ahora representa **dos**: la
red genera una palabra del doble de largo, la **parto por la mitad** y rearmo cada
mitad como un grafo, exactamente igual que antes. Así cada sesión produce un par
$(G_1, G_2)$.

Esto es el esqueleto: cuando tengamos pares, el objetivo de Stanley será buscar pares
**no isomorfos con el mismo $U_0$-polinomio**. Esa recompensa la defino en el próximo
paso; acá uso una provisional (que ambos grafos sean conexos) solo para que el ciclo
corra y se vea que la interpretación funciona.

> Este notebook duplica el algoritmo del notebook 01, así que los `CAMBIO 1-9 vs
> Wagner` siguen aplicando. Acá agrego los nuevos: **CAMBIO 10, 11 y 12** (ver
> `CAMBIOS_VS_WAGNER.md`).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import networkx as nx
import matplotlib.pyplot as plt
from numba import njit
import math, os, pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(0)
torch.manual_seed(0)
print("device:", device)

## Codificación: ahora la palabra son dos grafos

Cada grafo de $N$ vértices necesita $\binom{N}{2}$ bits. Para un par, la palabra mide
$2\binom{N}{2}$: la primera mitad es $G_1$ y la segunda es $G_2$. El estado que ve la
red sigue el mismo patrón de Wagner (palabra parcial + one-hot), o sea mide el doble:
$2 \cdot 2\binom{N}{2} = 4\binom{N}{2}$.

Uso $N = 7$ (como en el Enfoque 1 de la tesis, que corrió $N = 5, 6, 7$): grafos
chicos, fáciles de dibujar y rápidos de entrenar.

In [ ]:
N = 7                         # vertices por grafo (la tesis usa 5,6,7 en este enfoque)
MYN = N * (N - 1) // 2        # aristas por grafo

# >>> CAMBIO 10 vs Wagner: la palabra codifica DOS grafos, no uno
LARGO = 2 * MYN               # largo de la palabra = G1 (MYN) + G2 (MYN)
observation_space = 2 * LARGO # palabra parcial + one-hot, igual que Wagner pero del doble
len_game = LARGO              # una decision por cada arista de los dos grafos

n_sessions = 1000
PCT_ELITE = 93
PCT_SUPER = 94

# CAMBIO 3 (ver nb 01): Adam en vez de SGD
LR = 0.01
BETAS = (0.9, 0.999)
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

## Red neuronal

La misma del notebook 01 (128-64-4-1, ReLU, salida sigmoide). Lo único distinto es
que ahora la entrada mide $4\binom{N}{2}$ en vez de $2\binom{N}{2}$.

In [ ]:
# CAMBIO 1,2 (ver nb 01): framework PyTorch + nn.Module con init Xavier
class ModeloGrafo(nn.Module):
    def __init__(self, obs, n1=128, n2=64, n3=4):
        super().__init__()
        self.fc1 = nn.Linear(obs, n1)
        self.fc2 = nn.Linear(n1, n2)
        self.fc3 = nn.Linear(n2, n3)
        self.salida = nn.Linear(n3, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return torch.sigmoid(self.salida(x))

## Interpretación: partir la palabra en dos y rearmar cada grafo

Acá está el cambio central. Dada la palabra de largo $2\binom{N}{2}$, tomo la primera
mitad como $G_1$ y la segunda como $G_2$, y rearmo cada una con el mismo recorrido
triangular de siempre (el de Wagner). El resto del algoritmo no se entera.

In [ ]:
# >>> CAMBIO 11 vs Wagner: la palabra se parte en 2 mitades y cada una se rearma como un grafo

def palabra_a_grafo(g):
    # mismo recorrido triangular de Wagner, devuelve un grafo de networkx (para dibujar)
    G = nx.Graph(); G.add_nodes_from(range(N))
    c = 0
    for i in range(N):
        for j in range(i + 1, N):
            if g[c] == 1:
                G.add_edge(i, j)
            c += 1
    return G

def construir_par(palabra):
    # primera mitad -> G1, segunda mitad -> G2
    return palabra_a_grafo(palabra[:MYN]), palabra_a_grafo(palabra[MYN:])

@njit
def _palabra_a_adj(g, N):
    adj = np.zeros((N, N), dtype=np.int64)
    c = 0
    for i in range(N):
        for j in range(i + 1, N):
            if g[c] == 1:
                adj[i, j] = 1; adj[j, i] = 1
            c += 1
    return adj

# CAMBIO 5 (ver nb 01): conexidad por BFS en Numba
@njit
def num_componentes(adj):
    n = adj.shape[0]
    visto = np.zeros(n, dtype=np.uint8)
    cola = np.empty(n, dtype=np.int64)
    comp = 0
    for s in range(n):
        if not visto[s]:
            comp += 1
            qs = 0; qe = 0; cola[qe] = s; qe += 1; visto[s] = 1
            while qs < qe:
                u = cola[qs]; qs += 1
                for v in range(n):
                    if adj[u, v] == 1 and not visto[v]:
                        visto[v] = 1; cola[qe] = v; qe += 1
    return comp

## Demostración: ¿salen pares?

Con una red sin entrenar genero unas sesiones y rearmo los pares. Los grafos van a
salir feos (red aleatoria), pero lo que quiero mostrar es que la interpretación
**produce dos grafos por sesión**.

In [ ]:
def generar_sesiones(modelo, n_ses):
    estados = np.zeros((n_ses, observation_space, len_game), dtype=np.float32)
    acciones = np.zeros((n_ses, len_game), dtype=np.int8)
    estado = np.zeros((n_ses, observation_space), dtype=np.float32)
    estado[:, LARGO] = 1                       # one-hot arranca en el bloque de posicion (offset LARGO)
    for paso in range(len_game):
        with torch.no_grad():
            prob = modelo(torch.from_numpy(estado).to(device)).cpu().numpy().reshape(-1)
        acc = (np.random.rand(n_ses) < prob).astype(np.int8)
        estados[:, :, paso] = estado
        acciones[:, paso] = acc
        estado[:, paso] = acc
        estado[:, LARGO + paso] = 0
        if paso + 1 < len_game:
            estado[:, LARGO + paso + 1] = 1
    return estados, acciones

modelo_demo = ModeloGrafo(observation_space).to(device)
_, acc_demo = generar_sesiones(modelo_demo, 4)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for k in range(2):
    G1, G2 = construir_par(acc_demo[k])
    nx.draw_circular(G1, ax=axes[k, 0], node_size=140, with_labels=True)
    nx.draw_circular(G2, ax=axes[k, 1], node_size=140, with_labels=True)
    axes[k, 0].set_title(f"par {k} - G1 ({G1.number_of_edges()} aristas)")
    axes[k, 1].set_title(f"par {k} - G2 ({G2.number_of_edges()} aristas)")
plt.tight_layout()
os.makedirs("../figuras", exist_ok=True)
plt.savefig("../figuras/pares_demo.png", dpi=110)
plt.show()

## Recompensa (provisional)

El objetivo real de Stanley será que $G_1$ y $G_2$ sean **no isomorfos con el mismo
$U_0$-polinomio**. Eso lo armo en el próximo paso. Por ahora, para que el ciclo
entrene contra *algo* y la curva muestre aprendizaje, uso una recompensa provisional:
que ambos grafos sean **conexos y con pocas aristas**. Eso empuja hacia árboles, que
es justo el caso de la conjetura de Stanley (con sólo conexidad la recompensa se
satura al toque, porque un grafo aleatorio de pocos vértices casi siempre es conexo).

In [ ]:
# >>> CAMBIO 12 vs Wagner: la recompensa evalua el PAR, no un solo grafo
# PROVISIONAL: conexos y dispersos (empuja hacia arboles). La de Stanley (comparar U0) va despues.
@njit
def calcScore_par(palabra, N, MYN):
    a1 = _palabra_a_adj(palabra[:MYN], N)
    a2 = _palabra_a_adj(palabra[MYN:], N)
    c1 = num_componentes(a1)
    c2 = num_componentes(a2)
    e1 = np.sum(a1) // 2                    # aristas de G1
    e2 = np.sum(a2) // 2                    # aristas de G2
    # castigo desconexion fuerte + pocas aristas suave -> optimo en dos arboles
    return -float((c1 - 1) + (c2 - 1)) - 0.1 * float(e1 + e2)

def recompensas(acciones):
    r = np.empty(len(acciones))
    for j in range(len(acciones)):
        r[j] = calcScore_par(acciones[j].astype(np.int64), N, MYN)
    return r

## Entrenamiento

El ciclo es el mismo del notebook 01 (élite 93 / súper 94, BCE). No cambia nada: la
red ni se entera de que ahora la palabra son dos grafos, eso vive solo en la
recompensa y en la interpretación.

In [ ]:
def entrenar(iteraciones, n_ses=n_sessions, log_cada=1, carpeta="../datos"):
    os.makedirs(carpeta, exist_ok=True)
    modelo = ModeloGrafo(observation_space).to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=LR, betas=BETAS, weight_decay=WEIGHT_DECAY)
    bce = nn.BCELoss()

    sup_est = np.zeros((0, observation_space, len_game), dtype=np.float32)
    sup_acc = np.zeros((0, len_game), dtype=np.int8)
    sup_rec = np.zeros(0)
    hist_max, hist_top = [], []

    for it in range(iteraciones):
        est, acc = generar_sesiones(modelo, n_ses)
        rec = recompensas(acc)

        EST = np.concatenate([est, sup_est])
        ACC = np.concatenate([acc, sup_acc])
        REC = np.concatenate([rec, sup_rec])

        m_elite = REC >= np.percentile(REC, PCT_ELITE)
        m_super = REC >= np.percentile(REC, PCT_SUPER)

        # CAMBIO 8 (ver nb 01): paso de entrenamiento explicito
        X = EST[m_elite].transpose(0, 2, 1).reshape(-1, observation_space)
        y = ACC[m_elite].reshape(-1).astype(np.float32)
        opt.zero_grad()
        pred = modelo(torch.from_numpy(X).to(device))
        loss = bce(pred, torch.from_numpy(y).to(device).unsqueeze(1))
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), GRAD_CLIP)
        opt.step()

        sup_est, sup_acc, sup_rec = EST[m_super], ACC[m_super], REC[m_super]
        hist_max.append(float(REC.max()))
        hist_top.append(float(np.sort(REC)[-100:].mean()))

        if it % log_cada == 0:
            print(f"iter {it:4d} | mejor {REC.max():+.3f} | top100 {hist_top[-1]:+.3f} | loss {loss.item():.4f}")

    return modelo, hist_max, hist_top

In [ ]:
ITERACIONES = 30   # con la recompensa provisional converge rapido (ambos conexos)

modelo, hist_max, hist_top = entrenar(ITERACIONES)

In [ ]:
optimo = -0.1 * 2 * (N - 1)   # dos arboles (N-1 aristas c/u, ambos conexos)

plt.figure(figsize=(8, 4))
plt.plot(hist_max, label="mejor de la iteracion")
plt.plot(hist_top, label="promedio top-100")
plt.axhline(optimo, color="g", ls="--", lw=1, label="optimo (dos arboles)")
plt.xlabel("iteracion"); plt.ylabel("recompensa provisional")
plt.title(f"Generacion de pares (N={N}) - recompensa provisional")
plt.legend(); plt.tight_layout()
plt.savefig("../figuras/pares_recompensa_provisional.png", dpi=120)
plt.show()

## Próximo paso

Ya tenemos el algoritmo lanzando pares $(G_1, G_2)$. Lo que falta es la **recompensa
de Stanley**: premiar cuando $G_1$ y $G_2$ tengan el mismo $U_0$-polinomio pero **no**
sean isomorfos. Ahí es donde el algoritmo empieza realmente a buscar contraejemplos.